# GPT-4o

## 1.환경준비

### (1) 라이브러리 설치

In [ ]:
!pip install langchain langchain_core langchain-openai -q

### (2) 라이브러리 로딩

In [ ]:
import pandas as pd
import numpy as np
import os
from openai import OpenAI

from langchain_openai.chat_models import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage, AIMessage

### (3) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (4) OpenAI API Key 등록
* 환경변수로 key 등록

In [ ]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/langchain/'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

* ⚠️ 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
print(os.environ['OPENAI_API_KEY'][:40])

## 2.다양한 멀티모달 모델

### (1) STT (Speech to Text)
* GPT-4o-mini-transcribe

#### 1) OpenAI 클라이언트

* 파일 업로드

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 sample.m4a 선택

* 모델 사용하기

In [ ]:
client = OpenAI()

# mp3 파일 불러오기
with open("sample.m4a", "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",  # STT 모델
        file=f
    )

print("Transcribed Text:", transcript.text)

#### 2) LangChain 기반 실행

In [ ]:
from langchain_core.runnables import RunnableLambda
from openai import OpenAI

client = OpenAI()

def transcribe_audio(file_path: str) -> str:
    with open(file_path, "rb") as f:
        result = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=f
        )
    return result.text

# LangChain Runnable로 감싸기
transcriber = RunnableLambda(transcribe_audio)

# 실행
print(transcriber.invoke("sample.m4a"))


#### 3) 실습
* 여러분의 음성파일을 녹음합니다
    * 20초 이상
    * 대상 : 특정 스크립트를 읽거나, 일상 대화를 시도
    * 목소리, 속도, 톤 등을 다양하게 조절
* 모델에 넣고 텍스트로 잘 변환하는지 확인



### (2) TTS (Text to Speech)
* GPT-4o-mini-tts

#### 1) OpenAI 클라이언트

In [ ]:
from openai import OpenAI
from IPython.display import Audio

client = OpenAI()

# 텍스트 → 음성 변환
with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="alloy",
    input="안녕하세요, 코랩에서 재생하는 GPT-4o-mini-tts 예제입니다."
) as response:
    audio_bytes = response.read()   # 음성 데이터 가져오기

# IPython.display.Audio : Jupyter/Colab에서 바로 재생 가능
Audio(audio_bytes, rate=24000)

#### 2) LangChain 기반 실행

In [ ]:
client = OpenAI()

# TTS 함수 정의
def text_to_speech(text: str) -> bytes:
    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="coral",
        input=text
    ) as response:
        return response.read()

# Runnable로 감싸기
tts_runnable = RunnableLambda(text_to_speech)

# 실행
audio_bytes = tts_runnable.invoke("RunnableLambda로 감싼 TTS 예제입니다.")
Audio(audio_bytes, rate=24000)

#### 3) 실습
* 1~2 문장을 입력합니다.
* voice를 바꿔가며, 음성으로 잘 변환되는지 확인합니다.



### (3) Image → Text (이미지 설명 생성)
* GPT-4o-mini

#### 1) 인터넷 url로 이미지 입력

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# 이미지 입력 + 질문
msg = HumanMessage(content=[
    {"type": "text", "text": "이 이미지를 설명해줘."},
    {"type": "image_url", "image_url": {"url": "https://flexible.img.hani.co.kr/flexible/normal/600/337/imgdb/original/2025/0918/20250918502314.jpg"}}
])

result = llm.invoke([msg])
print("Image Caption:", result.content)

#### 2) 로컬 이미지 파일로 입력

* 파일 업로드

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 image_sonny.jpg 선택

* 모델 사용

In [ ]:
import base64
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# 로컬 이미지 읽어서 base64 변환
with open("image_sonny.jpg", "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()

msg = HumanMessage(content=[
    {"type": "text", "text": "이 이미지를 설명해줘."},
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
])

result = llm.invoke([msg])
print("Image Caption:", result.content)

#### 3) 실습
* 인터넷에서 다양한 이미지를 가져와서
* 다양한 질문을 수행합니다.


#### 4) base64 내용 보기

* base64인코딩 형태

In [ ]:
img_b64

* base64 코드를 이미지로 시각화

In [ ]:
from IPython.display import display, HTML

display(HTML(f"""
<figure style="text-align:left">
  <img src="data:image/png;base64,{img_b64}" style="max-width:560px;height:auto;" />
</figure>
"""))